# MultiRocket Experiments Notebook

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, GridSearchCV, GroupShuffleSplit
from sklearn.metrics import classification_report, f1_score, make_scorer
from sklearn.linear_model import RidgeClassifier, LogisticRegression

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
warnings.filterwarnings("ignore")

# Ensure sktime and skopt are available
try:
    import sktime
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "sktime", "scikit-optimize", "-q"])
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True

import sys
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception as e:
    pass

import data_utils
from base_utils_qwen import (
    SequenceExtractor,
    competition_scorer,
    evaluate_holdout,
    make_competition_scorer,
    prepare_bayesian_space
)
from multi_rocket_utils import (
    FlexibleMultiRocketClassifier,
    HierarchicalMultiRocketEnsemble
)

In [2]:
# ============================================================
# CONFIGURATION
# ============================================================
MODE = "single"  # "ensemble" or "single"
target_col = "bfrb"
orientation_col = "orientation"
search_mode = "grid"  # "grid" or "bayesian"

random_state = 42
n_splits = 3
n_iter = 10
train_size = 0.2  # Using 80% for training now instead of small balanced split
error_score_constant = 0.0
verbose = 3
do_cross_val = False

# Slicing options for Single Mode
slice_by_orientation = None  # e.g., ["Seated Straight", "Standing"]
slice_by_bfrb = None         # True for BFRB only, False for non-BFRB only, None for all

# Ensemble orientations split (Leave None to auto-split evenly)
l3_orientations = None 
l4_orientations = None 

if do_cross_val:
    cv_object = GroupKFold(n_splits=n_splits)
else:
    cv_object = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)

results_dir = Path("results")
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M")

# Scorer selection based on target
if target_col == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average="macro", zero_division=0)

print(f"MODE: {MODE}")
print(f"Search mode: {search_mode}")

MODE: single
Search mode: grid


In [ ]:
data_root = data_utils.find_data_root()
raw_train_df = pd.read_csv(data_root / "train.csv")
train_demo_df = pd.read_csv(data_root / "train_demographics.csv")

train_df = raw_train_df.set_index("row_id").copy(deep=True)

# Handedness & Upside-down corrections
if "handedness" in train_demo_df.columns:
    train_df["handedness"] = train_df["subject"].map(train_demo_df.set_index("subject")["handedness"])
    left_handed_mask = train_df["handedness"].eq(0)
    train_df.loc[left_handed_mask, "acc_x"] *= -1.0
    train_df = train_df.drop(columns=["handedness"])

upside_down_mask = train_df["subject"].isin(["SUBJ_019262", "SUBJ_045235"])
train_df.loc[upside_down_mask, ["acc_x", "acc_y", "acc_z"]] *= -1.0
train_df.loc[upside_down_mask, ["rot_x", "rot_y", "rot_z"]] *= -1.0

train_df["is_target"] = (train_df["sequence_type"] == "Target").astype(bool)
train_df["bfrb"] = train_df["gesture"].where(train_df["is_target"], "non_bfrb")

# Create alternative target columns
train_df["gesture_position"] = train_df["gesture"].str.split(" - ").str[0]
train_df["gesture_action"] = train_df["gesture"].str.split(" - ").str[-1]

Using local data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data


In [ ]:
# Get unique sequences for splitting to prevent leakage
sequences = train_df[['sequence_id', 'is_target', target_col, orientation_col]].drop_duplicates()
seq_ids = sequences['sequence_id'].unique()

if not do_cross_val:
    splitter = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(splitter.split(seq_ids, groups=seq_ids))
    train_seqs = seq_ids[train_idx]
    test_seqs = seq_ids[test_idx]
else:
    train_seqs = seq_ids # For CV, we use all data in the search object
    test_seqs = []

train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy() if len(test_seqs) > 0 else pd.DataFrame()

X_train = train_sample_df
X_test = hold_out_df

# Setup y based on MODE
if MODE == "ensemble":
    y_train = train_sample_df[["sequence_id", "is_target", orientation_col, target_col]].copy()
    y_test = hold_out_df[["sequence_id", "is_target", orientation_col, target_col]].copy() if len(hold_out_df) > 0 else pd.DataFrame()
else:
    y_train = train_sample_df[["sequence_id", target_col]].copy()
    y_test = hold_out_df[["sequence_id", target_col]].copy() if len(hold_out_df) > 0 else pd.DataFrame()

groups = X_train["sequence_id"]

print(f"Train sequences: {X_train['sequence_id'].nunique()} | Test sequences: {X_test['sequence_id'].nunique() if len(X_test) > 0 else 0}")

In [ ]:
# ============================================================
# PARAMETER SPACE DEFINITION
# ============================================================
if search_mode == "bayesian" and SKOPT_AVAILABLE:
    extractor_space = {
        "extractor__acc_modes": Categorical(["smoothed|velocity|displacement|jerk", "raw", 'raw|velocity', 'raw|jerk']),
        "extractor__rotation_modes": Categorical(["quaternion|angular_velocity|euler|delter_euler|rot6d", "raw", "quaternion", "quaternion|angular_velocity", "quaternion|euler"]),
        "extractor__tof_modes": Categorical(["pooled_stats|sensor_stats", 'raw', "pooled_stats", "sensor_stats"]),
        "extractor__thm_modes": Categorical(["centered_diff", 'raw', 'centered', 'centered_diff', 'diff']),
        # "extractor__sampling_rate": Integer(20, 200),
        # "extractor__maxlen": Integer(120, 200),
        # "extractor__window_size": Integer(10, 40),
        # "extractor__clip_value": Real(30.0, 100.0, prior="linear"),
        # "extractor__interp_mode": Categorical(["linear"]),
        # "extractor__motion_filter_mode": Categorical(["kalman", None]),
        # "extractor__kalman_process_noise": Real(1e-4, 1e-2, prior="log-uniform"),
        # "extractor__kalman_measurement_noise": Real(1e-2, 1e-1, prior="log-uniform"),
        # "extractor__padding_value": Categorical([0.0]), 
    }

    if MODE == "ensemble":
        model_space = {
            # "classifier__base_classifier_class": Categorical([RidgeClassifier, LogisticRegression]),
            
            # # Layer 1 (Binary)
            # "classifier__l1_num_kernels": Integer(1000, 2000),
            # "classifier__l1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l1_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            # "classifier__l1_class_weight": Categorical(["balanced", None]),
            
            # # Layer 2 (Orientation)
            # "classifier__l2_num_kernels": Integer(1000, 2000),
            # "classifier__l2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l2_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            # "classifier__l2_class_weight": Categorical(["balanced", None]),
            
            # # Layer 3: BFRB per Orientation (Models 1-4)
            # "classifier__l3_1_num_kernels": Integer(1000, 2000),
            # "classifier__l3_1_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_1_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            
            # "classifier__l3_2_num_kernels": Integer(1000, 2000),
            # "classifier__l3_2_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_2_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            
            # "classifier__l3_3_num_kernels": Integer(1000, 2000),
            # "classifier__l3_3_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_3_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            
            # "classifier__l3_4_num_kernels": Integer(1000, 2000),
            # "classifier__l3_4_alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__l3_4_feature_selection_percentile": Categorical([None, 25, 50, 75]),
            
            # "classifier__l3_class_weight": Categorical(["balanced", None]),
        }
        fixed_model_params = {
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
        }
    else:
        model_space = {
            # "classifier__base_classifier_class": Categorical([RidgeClassifier, LogisticRegression]),
            # "classifier__num_kernels": Integer(1000, 2000),
            # "classifier__alpha": Real(1e1, 1e4, prior="log-uniform"),
            # "classifier__feature_selection_percentile": Categorical([None, 25, 50, 75]),
            # "classifier__class_weight": Categorical(["balanced", None]),
        }
        fixed_model_params = {
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
        }

    param_space = {**extractor_space, **model_space, **fixed_model_params}
    param_space = prepare_bayesian_space(param_space) # Serialize dicts for skopt

else:  # GRID SEARCH (1 Candidate Default)
    param_space = {
        # Extractor defaults
        "extractor__acc_modes": ["smoothed|velocity|jerk"],
        "extractor__rotation_modes": ["quaternion|angular_velocity|euler"],
        "extractor__tof_modes": ["pooled_stats|sensor_stats"],
        "extractor__thm_modes": ["centered_diff"],
        "extractor__sampling_rate": [200],
        "extractor__maxlen": [160],
        "extractor__window_size": [40],
        "extractor__clip_value": [100.0],
        "extractor__interp_mode": ["linear"],
        "extractor__motion_filter_mode": ["kalman"],
        "extractor__kalman_process_noise": [1e-3],
        "extractor__kalman_measurement_noise": [1e-1],
        "extractor__padding_value": [0.0],
    }
    
    if MODE == "ensemble":
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            
            # Layer 1
            "classifier__l1_num_kernels": [2000],
            "classifier__l1_alpha": [1000.0],
            "classifier__l1_feature_selection_percentile": [50],
            "classifier__l1_class_weight": ["balanced"],
            
            # Layer 2
            "classifier__l2_num_kernels": [2000],
            "classifier__l2_alpha": [1000.0],
            "classifier__l2_feature_selection_percentile": [50],
            "classifier__l2_class_weight": ["balanced"],
            
            # Layer 3 (Models 1-4)
            "classifier__l3_1_num_kernels": [2000], "classifier__l3_1_alpha": [1000.0], "classifier__l3_1_feature_selection_percentile": [50],
            "classifier__l3_2_num_kernels": [2000], "classifier__l3_2_alpha": [1000.0], "classifier__l3_2_feature_selection_percentile": [50],
            "classifier__l3_3_num_kernels": [2000], "classifier__l3_3_alpha": [1000.0], "classifier__l3_3_feature_selection_percentile": [50],
            "classifier__l3_4_num_kernels": [2000], "classifier__l3_4_alpha": [1000.0], "classifier__l3_4_feature_selection_percentile": [50],
            "classifier__l3_class_weight": ["balanced"],
            
            # Fixed
            "classifier__orientation_col": [orientation_col],
            "classifier__target_col": [target_col],
        })
    else:
        param_space.update({
            "classifier__base_classifier_class": [RidgeClassifier],
            "classifier__num_kernels": [2000],
            "classifier__alpha": [1000.0],
            "classifier__feature_selection_percentile": [50],
            "classifier__class_weight": ["balanced"],
            "classifier__target_col": [target_col],
            "classifier__orientation_col": [orientation_col],
            "classifier__slice_by_orientation": [slice_by_orientation],
            "classifier__slice_by_bfrb": [slice_by_bfrb],
        })

print("Parameter Space Defined.")

In [ ]:
pipe = Pipeline([
    # We override padding_value to 0.0 here because MultiRocket convolutions react poorly to -999.0
    ("extractor", SequenceExtractor(padding_value=0.0)), 
    ("classifier", HierarchicalMultiRocketEnsemble(padding_value=0.0) if MODE == "ensemble" else FlexibleMultiRocketClassifier(padding_value=0.0))
])

if search_mode == "bayesian" and SKOPT_AVAILABLE:
    search = BayesSearchCV(
        pipe, param_space, n_iter=n_iter, scoring=scorer,
        cv=cv_object, n_jobs=1, random_state=random_state,
        error_score=error_score_constant, verbose=verbose, return_train_score=True
    )
else:
    search = GridSearchCV(
        pipe, param_space, scoring=scorer,
        cv=cv_object, n_jobs=1, error_score=error_score_constant, 
        verbose=verbose, return_train_score=True
    )

print(f"Running {search_mode} search...")
search.fit(X_train, y_train, groups=groups)

print(f"\nBest CV score: {search.best_score_:.4f}")
print("Best parameters:")
print(search.best_params_)

best_model = search.best_estimator_

In [ ]:
if not do_cross_val and len(X_test) > 0:
    y_pred = best_model.predict(X_test)
    
    # Collapse to sequence level
    y_test_seq = y_test.drop_duplicates("sequence_id").sort_values("sequence_id")
    
    if target_col == 'bfrb':
        eval_dict = evaluate_holdout(y_test_seq, y_pred, target_col=target_col, verbose=True)
        print(f"Holdout Competition Score: {eval_dict['competition_score']:.4f}")
    else:
        print("\nClassification Report:")
        print(classification_report(y_test_seq[target_col], y_pred, zero_division=0))
        macro_f1 = f1_score(y_test_seq[target_col], y_pred, average='macro', zero_division=0)
        print(f"Macro F1 Score: {macro_f1:.4f}")
else:
    print("Holdout evaluation skipped (Cross-Val mode or empty test set).")